# Scrollytelling Data Aggregation

This notebook reads `NammaMetro_Ridership_Dataset.csv`, filters to the editorial window (2024-10-26 to 2025-05-05), and emits 5 JSON files into `scrolly-data/` for the scrollytelling web app to consume.

The web app does no data wrangling — it just reads JSONs and renders.

## Outputs

1. `daily-by-mode.json` — daily ridership per payment channel + total
2. `mode-shares.json` — same shape, each field as a fraction of the day
3. `significant-events.json` — events from `significant_dates.csv` filtered to the window
4. `anomalies.json` — days where any channel's value diverges from its 7-day moving average by >30%
5. `stations.geojson` — station data from `NammaMetro_Stations_Dataset.csv` (copied)

In [11]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

DATA_DIR = Path('/Users/home/DEV/MY PROJECTS/namma-metro-ridership-tracker')
OUT_DIR = DATA_DIR / 'scrolly-article/'
OUT_DIR.mkdir(parents=True, exist_ok=True)

EDITORIAL_START = pd.Timestamp('2024-10-26')
EDITORIAL_END = pd.Timestamp('2025-05-05')

df = pd.read_csv(DATA_DIR / 'NammaMetro_Ridership_Dataset.csv')
df['date'] = pd.to_datetime(df['Record Date'], format='%d-%m-%Y')
df = df.sort_values('date').reset_index(drop=True)
print(f'Loaded {len(df)} rows: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Editorial window: {EDITORIAL_START.date()} → {EDITORIAL_END.date()}')

window = df[(df['date'] >= EDITORIAL_START) & (df['date'] <= EDITORIAL_END)].copy()
print(f'Rows in window: {len(window)}')

Loaded 438 rows: 2024-10-26 → 2026-07-15
Editorial window: 2024-10-26 → 2025-05-05
Rows in window: 150


## 1. Daily ridership by mode

In [12]:
CHANNELS = {
    'Total Smart Cards': 'smartcard',
    'Total Tokens': 'token',
    'Total NCMC': 'ncmc',
    'Group Ticket': 'groupTicket',
    'QR NammaMetro': 'qrNammaMetro',
    'QR WhatsApp': 'qrWhatsApp',
    'QR Paytm': 'qrPaytm',
}

by_mode = pd.DataFrame({'date': window['date'].dt.strftime('%Y-%m-%d')})
for src, dst in CHANNELS.items():
    by_mode[dst] = window[src].astype(int).values
by_mode['total'] = window[list(CHANNELS)].sum(axis=1).astype(int).values

by_mode.to_json(OUT_DIR / 'daily-by-mode.json', orient='records', indent=2)
print(f'Wrote daily-by-mode.json: {len(by_mode)} rows')
by_mode.head()

Wrote daily-by-mode.json: 150 rows


,date,smartcard,token,ncmc,groupTicket,qrNammaMetro,qrWhatsApp,qrPaytm,total
0,2024-10-26,353460,241883,7444,512,49351,95571,32357,780578
1,2024-10-27,176278,250124,4278,458,56321,111959,37630,637048
2,2024-10-28,452257,208014,11076,160,49051,94150,34428,849136
3,2024-10-29,452299,205421,10948,407,53315,94739,34156,851285
4,2024-10-30,425759,233366,9975,198,51942,111291,38350,870881


## 2. Mode shares per day

Each channel's share of the day's total ridership. Useful for the stacked-area chart in chapter 2 ("The seven doors") and chapter 8 ("Fare Hike Impact").

In [13]:
shares = by_mode.copy()
for src, dst in CHANNELS.items():
    shares[dst] = (shares[dst] / shares['total']).round(4)
shares = shares.drop(columns=['total'])

shares.to_json(OUT_DIR / 'mode-shares.json', orient='records', indent=2)
print(f'Wrote mode-shares.json: {len(shares)} rows')
shares.head()

Wrote mode-shares.json: 150 rows


,date,smartcard,token,ncmc,groupTicket,qrNammaMetro,qrWhatsApp,qrPaytm
0,2024-10-26,0.4528,0.3099,0.0095,0.0007,0.0632,0.1224,0.0415
1,2024-10-27,0.2767,0.3926,0.0067,0.0007,0.0884,0.1757,0.0591
2,2024-10-28,0.5326,0.2450,0.0130,0.0002,0.0578,0.1109,0.0405
3,2024-10-29,0.5313,0.2413,0.0129,0.0005,0.0626,0.1113,0.0401
4,2024-10-30,0.4889,0.2680,0.0115,0.0002,0.0596,0.1278,0.0440


## 3. Significant events

Events from `significant_dates.csv` filtered to the editorial window. Categorised by type: `fare_hike`, `festival`, `civic`, `concert`, `bandh`, `sports`.

In [14]:
events = pd.read_csv(DATA_DIR / 'significant_dates.csv')
events['date'] = pd.to_datetime(events['Date'], format='%Y-%m-%d')
events = events[(events['date'] >= EDITORIAL_START) & (events['date'] <= EDITORIAL_END)].copy()

def classify(label: str) -> tuple[str, str]:
    """Return (type, weight) for a given event label."""
    l = label.lower()
    if 'fare hike' in l:
        return 'fare_hike', 'primary'
    if 'concert' in l:
        return 'concert', 'primary'
    if 'bandh' in l:
        return 'bandh', 'secondary'
    if 'aero india' in l:
        return 'civic', 'primary'
    if any(k in l for k in ('diwali', 'holi', 'christmas', 'eid', 'ugadi', 'sankranti', 'pongal', 'onam', 'dasara', 'dussehra', 'navratri', 'republic day', 'independence day', 'may day', 'new year', 'valentine', 'rajayotsava', 'mahashivratri', 'ganesh', 'easter', 'good friday', 'gandhi', 'basava', 'ambedkar', 'mahavir', 'kanakadasa', 'gurupadavwa', 'varalakshmi', 'muharram', 'milad', 'equinox', 'solstice', 'bandh', 'marathon', 'champions trophy', 'asia cup', 't20 world cup', 'ipl', 'ranji', 'rcb', 'cricket', 'match', 'concert', 'aero india', 'toyota', 'lunar', 'mahalaya', 'karnataka')):
        return 'festival', 'secondary'
    return 'other', 'secondary'

events[['type', 'weight']] = events['Event'].apply(lambda l: pd.Series(classify(l)))
events_out = events[['date', 'Event', 'type', 'weight']].copy()
events_out['date'] = events_out['date'].dt.strftime('%Y-%m-%d')
events_out = events_out.rename(columns={'Event': 'label'})

events_out.to_json(OUT_DIR / 'significant-events.json', orient='records', indent=2)
print(f'Wrote significant-events.json: {len(events_out)} events')
events_out.head(20)

Wrote significant-events.json: 30 events


,date,label,type,weight
0,2024-10-31,Diwali,festival,secondary
1,2024-11-01,Rajayotsava Day,festival,secondary
2,2024-12-24,Christmas Eve,festival,secondary
3,2024-12-25,Christmas Day,festival,secondary
4,2024-12-31,New Year's Eve,festival,secondary
5,2025-01-01,New Year's Day,festival,secondary
6,2025-01-14,Makara Sankranti,festival,secondary
7,2025-01-25,Ranji Trophy (Bangalore),festival,secondary
8,2025-01-26,Republic Day,festival,secondary
9,2025-02-08,Ed Sheeran Concert,concert,primary


## 4. Anomaly detection

A day is flagged as anomalous if any channel's value diverges from its 7-day moving average by more than 30%. The baseline is computed using the 7 days *before* the day in question (so the day itself doesn't contaminate the baseline). For the first 7 days in the window, we use whatever history is available.

The two-direction rule catches:
- **Drops** — open-loop payment channels failing (e.g. Jan 15 Tokens, Jan 16 QR)
- **Surges** — closed-loop channels gaining (e.g. Jan 15-16 Smart Card)

The conspiracy chapter (9) uses these to investigate the Jan 15-16 sequence.

In [15]:
anomalies = []
for src, dst in CHANNELS.items():
    series = window[['date', src]].copy()
    series['ma7'] = series[src].rolling(window=7, min_periods=1).mean().shift(1)
    series['pct_change'] = ((series[src] - series['ma7']) / series['ma7'] * 100).round(2)
    flagged = series[series['pct_change'].abs() > 30].copy()
    for _, row in flagged.iterrows():
        anomalies.append({
            'date': row['date'].strftime('%Y-%m-%d'),
            'channel': dst,
            'value': int(row[src]),
            'baseline': round(float(row['ma7']), 1) if pd.notna(row['ma7']) else None,
            'changePct': float(row['pct_change']),
        })

anomalies_df = pd.DataFrame(anomalies).sort_values(['date', 'channel']).reset_index(drop=True)
anomalies_df.to_json(OUT_DIR / 'anomalies.json', orient='records', indent=2)
print(f'Wrote anomalies.json: {len(anomalies_df)} anomalies across the {len(CHANNELS)} channels')
print(f'\nAnomalies on Jan 15-16 2025 (the smoking gun):')
print(anomalies_df[anomalies_df['date'].isin(['2025-01-15', '2025-01-16'])])
print(f'\nAnomalies on Dec 6 2024 (the QR NammaMetro drop):')
print(anomalies_df[anomalies_df['date'] == '2024-12-06'])

Wrote anomalies.json: 231 anomalies across the 7 channels

Anomalies on Jan 15-16 2025 (the smoking gun):
           date       channel   value  baseline  changePct
102  2025-01-15   groupTicket     228     386.7     -41.04
103  2025-01-15          ncmc   14210   10665.7      33.23
104  2025-01-15     smartcard  543080  369758.0      46.87
105  2025-01-15         token   84841  186421.3     -54.49
106  2025-01-16   groupTicket     205     350.4     -41.50
107  2025-01-16          ncmc   15026   10971.1      36.96
108  2025-01-16  qrNammaMetro    3947   51260.9     -92.30
109  2025-01-16       qrPaytm    2285   34946.7     -93.46
110  2025-01-16    qrWhatsApp    6671  102143.9     -93.47
111  2025-01-16     smartcard  631442  389181.6      62.25

Anomalies on Dec 6 2024 (the QR NammaMetro drop):
          date       channel  value  baseline  changePct
52  2024-12-06   groupTicket    219     708.7     -69.10
53  2024-12-06  qrNammaMetro   1081   51886.3     -97.92


## 5. Stations

Copied from `NammaMetro_Stations_Dataset.csv` and converted to GeoJSON. The line-color emoji codes are mapped to hex colors for the map chapter (1).

In [16]:
LINE_COLORS = {
    '🟣': '#7e3eb5',  # Purple Line
    '🟢': '#4caf50',  # Green Line
    '🟡': '#fbc02d',  # Yellow Line (future)
    '🔵': '#2196f3',  # Blue Line (future)
    '🟠': '#ff9800',  # Orange Line (future)
    '🔴': '#e53935',  # Red Line (future)
}

stations = pd.read_csv(DATA_DIR / 'NammaMetro_Stations_Dataset.csv')
print(f'Loaded {len(stations)} stations')
print(stations.head())
print(f'\nColumns: {list(stations.columns)}')

# Note: the existing CSV doesn't have lat/lng — just station metadata.
# For the scrolly map, we'll use a hand-coded schematic of the two operational lines.
# This is a placeholder GeoJSON with the station list and line colors, but no coordinates.
# The actual line geometry will be hardcoded in the scrolly component.

features = []
for _, row in stations.iterrows():
    features.append({
        'type': 'Feature',
        'properties': {
            'code': row['code'],
            'name': row['name_eng'],
            'name_kan': row['name_kan'],
            'lines': list(row['lines']) if isinstance(row['lines'], str) else [],
            'lineColors': [LINE_COLORS.get(c, '#999') for c in (row['lines'] if isinstance(row['lines'], str) else [])],
            'isTerminus': bool(row['is_terminus']),
            'isInterchange': bool(row['is_interchange']),
        },
        'geometry': None,  # coordinates to be hand-coded in the scrolly component
    })

geojson = {
    'type': 'FeatureCollection',
    'features': features,
    'metadata': {
        'lines': {emoji: hex_color for emoji, hex_color in LINE_COLORS.items()},
        'note': 'Geometry is null in this file. The scrolly map (Chapter 1) hand-codes the line paths.',
    },
}

with open(OUT_DIR / 'stations.geojson', 'w') as f:
    json.dump(geojson, f, indent=2, ensure_ascii=False)
print(f'\nWrote stations.geojson: {len(features)} features')

Loaded 172 stations
    code        name_eng           name_kan lines  is_terminus  is_interchange
0  [TBC]           Agara                ಅಗರ    🔵🔴        False            True
1  [TBC]    Airport City  ವಿಮಾನ ನಿಲ್ದಾಣ ನಗರ     🔵        False           False
2  [TBC]  Ambedkar Nagar      ಅಂಬೇಡ್ಕರ್ ನಗರ     🔴        False           False
3   AGPP       Attiguppe        ಅತ್ತಿಗುಪ್ಪೆ     🟣        False           False
4  [TBC]      BEL Circle     ಬಿ ಇ ಎಲ್ ವೃತ್ತ     🟠        False           False

Columns: ['code', 'name_eng', 'name_kan', 'lines', 'is_terminus', 'is_interchange']

Wrote stations.geojson: 172 features


## 6. Hypothesis-testing window (for Chapter 9)

The notebook's chapter 9 uses 'before vs after Jan 14' without specifying the window. The scrolly uses **14 days each side** of Jan 14, per sign-off. The 14-day pre-period and 14-day post-period are exported for the chapter 9 H₀/H₁ test to use directly.

In [17]:
PIVOT = pd.Timestamp('2025-01-14')
WINDOW_DAYS = 14
pre_start = PIVOT - pd.Timedelta(days=WINDOW_DAYS - 1)  # 14 days inclusive of Jan 14? No — we exclude Jan 14 itself.
pre_start = PIVOT - pd.Timedelta(days=WINDOW_DAYS)
pre_end = PIVOT - pd.Timedelta(days=1)
post_start = PIVOT + pd.Timedelta(days=1)
post_end = PIVOT + pd.Timedelta(days=WINDOW_DAYS)

pre = by_mode[(by_mode['date'] >= pre_start.strftime('%Y-%m-%d')) & (by_mode['date'] <= pre_end.strftime('%Y-%m-%d'))].copy()
post = by_mode[(by_mode['date'] >= post_start.strftime('%Y-%m-%d')) & (by_mode['date'] <= post_end.strftime('%Y-%m-%d'))].copy()
print(f'Pre-period: {pre["date"].iloc[0]} → {pre["date"].iloc[-1]} ({len(pre)} days)')
print(f'Post-period: {post["date"].iloc[0]} → {post["date"].iloc[-1]} ({len(post)} days)')

hypothesis_window = {
    'pivot': '2025-01-14',
    'pivotLabel': 'Jan 14, 2025 (Sankranti / day before the anomaly)',
    'windowDays': WINDOW_DAYS,
    'pre': {
        'start': pre_start.strftime('%Y-%m-%d'),
        'end': pre_end.strftime('%Y-%m-%d'),
        'days': pre['date'].tolist(),
    },
    'post': {
        'start': post_start.strftime('%Y-%m-%d'),
        'end': post_end.strftime('%Y-%m-%d'),
        'days': post['date'].tolist(),
    },
    'hypothesis': {
        'H0': 'There is no significant difference in the mean transaction volume before and after Jan 14 for any payment method. Any observed difference is due to random variation.',
        'H1': 'There is a significant difference in the mean transaction volume before and after Jan 14 for any payment method. The difference indicates a systematic shift in commuter behaviour.',
    },
}

with open(OUT_DIR / 'hypothesis-window.json', 'w') as f:
    json.dump(hypothesis_window, f, indent=2)
print(f'\nWrote hypothesis-window.json')
print(f'Pre days:  {len(hypothesis_window["pre"]["days"])}')
print(f'Post days: {len(hypothesis_window["post"]["days"])}')

Pre-period: 2024-12-31 → 2025-01-08 (8 days)
Post-period: 2025-01-15 → 2025-01-28 (14 days)

Wrote hypothesis-window.json
Pre days:  8
Post days: 14


## 7. Fare-hike window (for Chapter 8)

The notebook's chapter 8 uses a 6-week window centered on Feb 9. The scrolly extends to 12 weeks (3 weeks before, 12 weeks after — full data).

In [18]:
FARE_HIKE = pd.Timestamp('2025-02-09')
FARE_HIKE_PRE_DAYS = 21   # 3 weeks before
FARE_HIKE_POST_DAYS = 84  # 12 weeks after (full data)

fh_pre_start = FARE_HIKE - pd.Timedelta(days=FARE_HIKE_PRE_DAYS)
fh_post_end = FARE_HIKE + pd.Timedelta(days=FARE_HIKE_POST_DAYS)
fh_pre = by_mode[(by_mode['date'] >= fh_pre_start.strftime('%Y-%m-%d')) & (by_mode['date'] < FARE_HIKE.strftime('%Y-%m-%d'))].copy()
fh_post = by_mode[(by_mode['date'] > FARE_HIKE.strftime('%Y-%m-%d')) & (by_mode['date'] <= fh_post_end.strftime('%Y-%m-%d'))].copy()
print(f'Fare-hike window: {FARE_HIKE.date()}')
print(f'Pre:  {fh_pre["date"].iloc[0]} → {fh_pre["date"].iloc[-1]} ({len(fh_pre)} days)')
print(f'Post: {fh_post["date"].iloc[0]} → {fh_post["date"].iloc[-1]} ({len(fh_post)} days)')

fare_hike_window = {
    'pivot': '2025-02-09',
    'pivotLabel': 'Feb 9, 2025 (Fare hike)',
    'preDays': FARE_HIKE_PRE_DAYS,
    'postDays': FARE_HIKE_POST_DAYS,
    'pre': {
        'start': fh_pre_start.strftime('%Y-%m-%d'),
        'end': fh_pre['date'].iloc[-1],
        'days': fh_pre['date'].tolist(),
    },
    'post': {
        'start': fh_post['date'].iloc[0],
        'end': fh_post['date'].iloc[-1],
        'days': fh_post['date'].tolist(),
    },
}

with open(OUT_DIR / 'fare-hike-window.json', 'w') as f:
    json.dump(fare_hike_window, f, indent=2)
print(f'\nWrote fare-hike-window.json')

Fare-hike window: 2025-02-09
Pre:  2025-01-19 → 2025-02-08 (20 days)
Post: 2025-02-10 → 2025-05-04 (58 days)

Wrote fare-hike-window.json


## 8. Done — what's in `scrolly-article/`

In [19]:
for f in sorted(OUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:30s} {size_kb:8.1f} KB')

  aggregate-scrolly.ipynb            35.3 KB
  anomalies.json                     28.1 KB
  daily-by-mode.json                 30.6 KB
  fare-hike-window.json               1.8 KB
  hypothesis-window.json              1.1 KB
  mode-shares.json                   28.6 KB
  significant-events.json             3.3 KB
  stations.geojson                   64.4 KB
